In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast
from pathlib import Path
import torch
from typing import List, Optional
from dataclasses import dataclass

import teradatasql
from sqlalchemy import text, create_engine
from teradataml import create_context, get_context, get_connection, DataFrame, in_schema, copy_to_sql
from teradataml.dataframe.copy_to import copy_to_sql
from dotenv import load_dotenv

import torch
from sentence_transformers import SentenceTransformer

# import sys
# sys.path.append('..')
from constants import (
    CLEANED_TEST_DATA_PATH,
    ENCODED_TEST_DATA_PATH,
    CLEANED_TRAIN_DATA_PATH
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mk255155\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# load_dotenv('../.env')

# TD_HOST = os.getenv('TD_HOST')
# TD_USER = os.getenv('TD_USER')
# TD_PASS = os.getenv('TD_PASS')
# TD_DB = os.getenv('TD_DB')

TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"
TD_DB="DEMO_USER"

In [ ]:
# print(TD_DB,TD_HOST,TD_PASS,TD_USER)
# conn = teradatasql.connect(
#     host=TD_HOST,
#     user=TD_USER,
#     password=TD_PASS,
#     database=TD_DB
# )
# cursor = conn.cursor()
# print("Successfully connected to Teradata!")

In [3]:
conn = teradatasql.connect(
    host=TD_HOST,
    user=TD_USER,
    password=TD_PASS,
    logdata={'CHARSET': 'UTF8'}
)

sqlalchemy_engine = create_engine("teradatasql://", creator=lambda: conn)
create_context(tdsqlengine=sqlalchemy_engine)
print("Connection successful with UTF-8 encoding!")

Connection successful with UTF-8 encoding!


In [ ]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

In [ ]:
tdf = DataFrame.from_table("original_dataset", schema_name=TD_DB, index_label="Item_Name")
print("Shape of the data:", tdf.shape)

In [ ]:
tdf.head(10)

In [ ]:
tdf.tdtypes

In [ ]:
tdf = tdf.dropna(subset=["Item_Name", "class"])

In [ ]:
tdf.count()

In [ ]:
tdf = tdf.assign(
    Item_Name = tdf.Item_Name.str.lower(),
    **{'class': tdf['class'].str.lower()}
)

In [ ]:
tdf_stripped = tdf.assign(
    Item_Name = tdf.Item_Name.str.strip(),
    **{'class': tdf['class'].str.strip()}
)

In [ ]:
tdf = tdf_stripped.assign(
    Item_Name = tdf_stripped.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", ""),
    **{'class': tdf_stripped['class'].otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")}
)

In [ ]:
cleaned_tdf = tdf[["Item_Name", "class"]]
cleaned_tdf

In [ ]:
cleaned_tdf = cleaned_tdf.sort('row_id')
cleaned_tdf.head(10)

In [ ]:
from teradataml import execute_sql
from teradatasqlalchemy.types import VARCHAR

target_table_name = 'DEMO_USER.cleaned_data'

unicode_type = VARCHAR(length=500, charset='UNICODE')

unicode_tdf = cleaned_tdf.assign(
    Item_Name = cleaned_tdf.Item_Name.cast(unicode_type)
)

source_query = unicode_tdf.show_query()

create_sql = f"""
CREATE TABLE {target_table_name} AS (
    {source_query}
) WITH DATA;
"""

print("--- Generated SQL ---")
print(create_sql)

try:
    print(f"\nAttempting to drop existing table '{target_table_name}'...")
    execute_sql(f"DROP TABLE {target_table_name};")
    print("✔ Previous table dropped.")
except Exception:
    print("Table did not exist, proceeding to create.")

print("\nExecuting CREATE TABLE statement... 🚀")
execute_sql(create_sql)
print(f"✔ Success! Table '{target_table_name}' has been created.")

# Retrieving Cleaned Data from database and creating Embeddings

In [4]:
tdf_embedd = DataFrame.from_table("cleaned_data", schema_name=TD_DB, index_label="row_id")
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (4772, 3)


row_id,Item_Name,class
3,أبو كاس أرز مزة بسمتي هندي 10 كجم,rice pasta pulses
5,أجنحة دجاج أطياب 700جم,poultry
4,أجرومونتي صلصة الداترينو 330 جم,tins jars packets
2,أبو عوف قهوة تركي محوج وسط 250 جم,tea coffee hot drinks
1,أبو علي بابريكا 85جم,cooking ingredients


In [ ]:
unique_classes_tdf = tdf_embedd.assign(
    drop_columns=True,
    class_name=tdf_embedd['class'].distinct()
)
unique_classes_tdf.count()

In [ ]:
unique_classes_tdf = tdf_embedd.drop_duplicate(column_names='class')
unique_classes_tdf.count()

In [ ]:
from teradataml import execute_sql
from teradatasqlalchemy.types import VARCHAR

target_table_name = 'DEMO_USER.unique_classes'

# Define the unicode type for the 'class' column.
unicode_type = VARCHAR(length=500, charset='UNICODE')

# This part is correct.
unicode_unique_classes_tdf = unique_classes_tdf.assign(
    **{'class': unique_classes_tdf['class'].cast(unicode_type)}
)

source_query = unicode_unique_classes_tdf.show_query()

# --- SQL Statement Generation ---

# 1. DDL with the "class" column quoted and a PRIMARY INDEX defined.
create_sql = f"""
CREATE MULTISET TABLE {target_table_name} (
    class_id INTEGER GENERATED ALWAYS AS IDENTITY (
        START WITH 1
        INCREMENT BY 1
        MINVALUE 1
        NO MAXVALUE
        NO CYCLE
    ),
    "class" VARCHAR(500) CHARACTER SET UNICODE
) PRIMARY INDEX (class_id);
"""

# 2. DML with the "class" column quoted.
insert_sql = f"""
INSERT INTO {target_table_name} ("class")
{source_query};
"""

print("--- Generated CREATE TABLE SQL ---")
print(create_sql)
print("\n--- Generated INSERT INTO SQL ---")
print(insert_sql)

# --- Execution ---

# Before creating the new table, attempt to drop it if it already exists.
try:
    print(f"\nAttempting to drop existing table '{target_table_name}'...")
    execute_sql(f"DROP TABLE {target_table_name};")
    print("✔ Previous table dropped.")
except Exception:
    print("Table did not exist, proceeding to create.")

# Execute the CREATE TABLE statement first to build the table structure.
print("\nExecuting CREATE TABLE statement... 🚀")
execute_sql(create_sql)
print(f"✔ Success! Table '{target_table_name}' has been created with an auto-incrementing ID.")

# Execute the INSERT statement to populate the table with your data.
print("\nExecuting INSERT statement to populate data... 📊")
execute_sql(insert_sql)
print(f"✔ Success! Data has been inserted into '{target_table_name}'.")

In [5]:
classes_tdf = DataFrame.from_table("unique_classes", schema_name=TD_DB)
print("Shape of the data:", classes_tdf.shape)
classes_tdf.head(30)

Shape of the data: (32, 2)


class_id,class
3,sauces dressings condiments
5,dairy eggs
6,cleaning supplies
7,tea coffee hot drinks
10002,biscuits cakes
10003,bakery
10004,poultry
10005,home textile
10006,beef processed meat
10007,nuts dates dried fruits


In [6]:
pandas_df = tdf_embedd.to_pandas()
pandas_df.head(10)

,Item_Name,class
row_id,,
1,أبو علي بابريكا 85جم,cooking ingredients
299,المصرين مكرونه فرن350جم,rice pasta pulses
1785,شويبس جولد خوخ زجاج,soft drinks juices
3172,نسكويك حليب بطعم الفراولة 180 مل,dairy eggs
2,أبو عوف قهوة تركي محوج وسط 250 جم,tea coffee hot drinks
300,المطبخ ارز مصرى 5,rice pasta pulses
1786,شويبس جولد مشروب شعير أناناس، 1 لتر,soft drinks juices
3173,نسكويك مشروب شيكولاتة 11 جم,tea coffee hot drinks
3,أبو كاس أرز مزة بسمتي هندي 10 كجم,rice pasta pulses


In [7]:
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Using device: {device}")
print("-" * 30)

Using device: cuda
------------------------------


In [8]:
# --- 3. Embed the "Item_Name" Column (Corrected for Final Format) ---

item_texts = pandas_df["Item_Name"].fillna("").astype(str).tolist()
item_texts_for_model = [f"passage: {text}" for text in item_texts]
print(f"Embedding {len(item_texts_for_model)} item names...")

item_embeddings = model.encode(
    item_texts_for_model,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = item_embeddings.shape[1]
emb_cols = [f"v{i}" for i in range(dim)]
item_embeddings_df_vectors = pd.DataFrame(item_embeddings, columns=emb_cols)


identifiers_df = pd.DataFrame({'row_id': pandas_df.index})

# Combine the identifiers with the embeddings
item_embeddings_df = pd.concat([identifiers_df, item_embeddings_df_vectors], axis=1)

print("\n✅ DataFrame with Item_Name Embeddings (item_embeddings_df):")
print(item_embeddings_df.head())

Embedding 4772 item names...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]


✅ DataFrame with Item_Name Embeddings (item_embeddings_df):
   row_id        v0        v1        v2        v3        v4        v5  \
0       1  0.010319  0.015229 -0.005427 -0.050385  0.001117  0.001965   
1     299  0.017769  0.024794 -0.008909 -0.044485 -0.000665  0.017517   
2    1785 -0.013817  0.021322 -0.004026 -0.032561  0.018748  0.004608   
3    3172  0.012451  0.019397  0.014731 -0.047708  0.001469  0.007881   
4       2  0.026756  0.023502 -0.000816 -0.060263  0.002704  0.011692   

         v6        v7        v8  ...     v1014     v1015     v1016     v1017  \
0  0.000756  0.050589  0.023118  ... -0.037539 -0.037331  0.023343 -0.028091   
1  0.005789  0.051188  0.034877  ... -0.029376 -0.028507  0.010227 -0.038087   
2 -0.006788  0.054900  0.031954  ... -0.036111 -0.048099 -0.011835 -0.023007   
3 -0.027354  0.053050  0.041065  ... -0.011765 -0.037892  0.022794 -0.039843   
4 -0.008044  0.037709  0.029987  ... -0.045107 -0.051180  0.012533 -0.038237   

      v1018     v10

In [ ]:
pandas_class_df = classes_tdf.to_pandas()
pandas_class_df.head()


In [ ]:
# --- 4. Embed the Unique "class" Column ---
unique_classes = pandas_class_df["class"].dropna().tolist()
class_texts_for_model = [f"query: {cls}" for cls in unique_classes]
print(f"\nEmbedding {len(class_texts_for_model)} unique classes...")

class_embeddings = model.encode(
    class_texts_for_model,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

class_embeddings_df = pd.DataFrame(class_embeddings, columns=emb_cols)
class_embeddings_df.insert(0, "class", unique_classes)

print("\n✅ DataFrame with Unique Class Embeddings (class_embeddings_df):")
print(class_embeddings_df.head())
print("-" * 30)


Embedding 32 unique classes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DataFrame with Unique Class Embeddings (class_embeddings_df):
                         class        v0        v1        v2        v3  \
0  chocolates sweets  desserts  0.040662  0.030359 -0.001194 -0.043742   
1              home appliances  0.045509  0.047589 -0.030944 -0.037738   
2     nuts dates  dried fruits  0.030891  0.023749 -0.042409 -0.039749   
3         disposables  napkins  0.031241  0.030066 -0.022194 -0.021763   
4                  dairy  eggs  0.034292  0.016726  0.000064 -0.041235   

         v4        v5        v6        v7        v8  ...     v1014     v1015  \
0  0.046111 -0.023970 -0.035002  0.031369  0.032979  ... -0.031254 -0.038728   
1  0.062515 -0.024960 -0.002550  0.053485  0.017696  ... -0.035716 -0.062548   
2  0.029337 -0.011062 -0.013213  0.058937  0.038234  ... -0.038736 -0.045293   
3  0.036281 -0.015769 -0.007946  0.058685  0.022922  ... -0.049391 -0.058889   
4  0.009469  0.009626 -0.006573  0.065538  0.025866  ... -0.037838 -0.033225   

      v10

In [ ]:
## Verify unicode on teradata session

# td_context = get_context()
# raw_connection = td_context.engine.raw_connection()
# print("Verifying the character set of the established session...")
# with raw_connection.cursor() as cur:
#     cur.execute("HELP SESSION;")
    
#     session_info = cur.fetchone()
    
#     if session_info:
#         character_set = session_info[5]
#         print(f"✅ Session Character Set is: {character_set}")
#     else:
#         print("❌ Could not retrieve session information.")

# raw_connection.close()

In [ ]:
# copy_to_sql(
#     df=cleaned_tdf,              
#     table_name="cleaned_data",
#     if_exists="replace"
# )

In [ ]:
# tdf_2 =  tdf[['Item_Name', 'class']]
# tdf_2
######
# cat=tdf.groupby(['class']).count()
# cat
#######

In [ ]:
## cleaning
# tdf = tdf.assign(Item_Name = tdf.Item_Name.str.lower())
# tdf_stripped = tdf.assign(Item_Name = tdf.Item_Name.str.strip())

# tdf = tdf_stripped.assign(
#     Item_Name = tdf.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")
# )

In [ ]:
# remove_context()